# R19-H210 - Conflict density at scale: does version diversity manifest?

Executor notebook (2026-07-07), **neo4j3 wave-2 graph, READ-ONLY, CPU-only, no LLM / no Bedrock**.

Reruns the H204 home-doc-restricted conflict-derivation harness on the completed wave-2 graph, WITH the
H208 dimension-aware unit-conversion comparator and alias-collapse. H204 found genuine conflicts rare on
the single-sourced product corpus (3 distinct, 1 genuine). H210 tests whether version-diverse articles at
scale (952 docs) manifest real conflicts and the silent-drop provenance failure.

- **clause (a)** - >= 40 distinct physical conflicts post alias-collapse and post unit-conversion
- **clause (b)** - silent-drop rate (graph carries one value / neither, vs both) stays >= 60%
- **refuted** if conflicts stay rare at scale (single-sourcing is then a corpus-class property)

The recall@8 retrieval arm of H204 is dropped here (it needs Bedrock query embeddings, banned by the H210
constraints); clauses (a) and (b) are pure text-derivation + graph-render census, no retrieval needed.

## Setup - CPU-only, neo4j3 pinned (config-apnea.yml authoritative)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""                       # CPU-only, no GPU
import re, json, hashlib, time, datetime, unicodedata, itertools
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from neo4j import GraphDatabase
from dotenv import dotenv_values
from knowledge_graph_foundry import load_settings
from rich import print as rprint

ROOT = Path("..")
STAMP = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
LOG = open(ROOT / "logs/h203-h210-scale.log", "a")
def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H210 {m}\n"); LOG.flush(); rprint(m)

settings = load_settings(ROOT / "config-apnea.yml")           # DEF-4 config-over-env precedence
NEO4J3 = settings.neo4j.uri
_env = dotenv_values(ROOT / ".env")
PW = settings.neo4j.password or _env.get("NEO4J_PASSWORD")
AUTH = (settings.neo4j.user, PW)
driver3 = GraphDatabase.driver(NEO4J3, auth=AUTH)             # READ-ONLY
log(f"config uri={NEO4J3} (CPU-only, neo4j3 pinned, READ-ONLY)")

2026-07-07 22:55:52.633 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


/tmp/ipykernel_2807372/1338885591.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  STAMP = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
/tmp/ipykernel_2807372/1338885591.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H210 {m}\n"); LOG.flush(); rprint(m)


config uri=bolt://user-konrad.jelen-kgf-neo4j3:7687 (CPU-only, neo4j3 pinned, READ-ONLY)

## Graph pull + fingerprint (H197 pattern) + render primitives (H204 verbatim)

In [2]:
with driver3.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN e.id AS id, e.name AS name, e.description AS description, "
                 "properties(e) AS props, labels(e) AS types").data()
    edges = s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id "
                  "RETURN DISTINCT a.id AS a, b.id AS b, type(r) AS rel").data()
    alias_rows = s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id "
                       "RETURN e.id AS eid, collect(DISTINCT a.id)[..5] AS aliases").data()
    emb_head = {r["id"]: r["head"] for r in s.run(
        "MATCH (e:Entity) WHERE e.embedding IS NOT NULL RETURN e.id AS id, e.embedding[0..8] AS head").data()}
    graph_docs = set(r["nm"] for r in s.run("MATCH (dd:KGFDocument) RETURN dd.name AS nm").data())
    n_docs = s.run("MATCH (dd:KGFDocument) RETURN count(dd) AS c").single()["c"]
node = {r["id"]: r for r in ents}; names = {r["id"]: r["name"] for r in ents}
alias_by = {r["eid"]: r["aliases"] for r in alias_rows}
rels_by = defaultdict(list)
for e in edges:
    rels_by[e["a"]].append((e["rel"], e["b"])); rels_by[e["b"]].append((e["rel"], e["a"]))
props_by = defaultdict(list)                                   # no Proposition nodes on wave-2 graph

def spec_of(r): return {k.removeprefix("prop_"): v for k, v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r = node[nid]; spec = dict(spec_of(r))
    for a in [a for a in alias_by.get(nid, []) if a in node]:
        for k, v in spec_of(node[a]).items(): spec.setdefault(k, v)
    return spec
def base_render(nid):
    r = node[nid]; spec = merged_spec(nid); al = [a for a in alias_by.get(nid, []) if a in node]
    aka = (f"Also known as: {', '.join(names.get(a, '') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid):
    return base_render(nid) + "\nRelations: " + "; ".join(f"{t} -> {names.get(b,'')}" for t, b in rels_by.get(nid, [])[:15])
def units_of(ids): return [seed_render(n) for n in ids] + [t for n in ids for t in props_by.get(n, [])]

def fingerprint():
    dig = ";".join(f"{i}:" + ",".join(f"{x:.4f}" for x in emb_head[i]) for i in sorted(emb_head))
    return dict(node_count=len(node), edge_count=len(edges), document_count=int(n_docs),
                embedding_count=len(emb_head),
                embedding_digest=hashlib.sha256(dig.encode()).hexdigest()[:16])
t0 = time.time(); FP = fingerprint()
log(f"fingerprint ({(time.time()-t0)*1000:.0f} ms): {json.dumps(FP)}")
log(f"graph: {len(node)} entities / {len(edges)} edges / {n_docs} docs / {len(emb_head)} embedded")

/tmp/ipykernel_2807372/1338885591.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H210 {m}\n"); LOG.flush(); rprint(m)


fingerprint (8 ms): {"node_count": 4649, "edge_count": 9410, "document_count": 952, "embedding_count": 4649, 
"embedding_digest": "8c4267c771d383bd"}

graph: 4649 entities / 9410 edges / 952 docs / 4649 embedded

## Source texts - all graph-indexed wave-2 articles (wave_01 + wave_02 on disk, restricted to graph docs)

The neo4j3 graph spans both ingest waves (c34 + c77 = 952 docs); both source dirs are on disk. Restrict to
docs present in the graph so the conflict derivation is graph-grounded.

In [3]:
_TM = dict.fromkeys(map(ord, "®™©"), None)
def gnorm(s):
    s = (s or "").translate(_TM); s = unicodedata.normalize("NFKC", s)
    s = s.replace(" ", " ").replace("×", "x").replace("*", "x").replace("·", "x")
    s = re.sub(r"(?<=\d),(?=\d)", "", s)
    return re.sub(r"\s+", " ", s.casefold()).strip()
def _n(s): return re.sub(r"\s+", " ", (s or "").casefold())
def toks(s): return set(re.findall(r"[a-z0-9]+", s.lower()))

rawd, title = {}, {}
for wave in ["wave_01", "wave_02"]:
    d = ROOT / "data/interim/apnea-waves" / wave
    for fp in d.iterdir():
        if fp.suffix == ".txt" and fp.name in graph_docs:
            txt = fp.read_text(errors="ignore")
            rawd[fp.name] = txt; title[fp.name] = _n(txt[:300])
DOCS = sorted(rawd)
log(f"source texts loaded: {len(DOCS)} graph-indexed articles (of {len(graph_docs)} graph docs)")

/tmp/ipykernel_2807372/1338885591.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H210 {m}\n"); LOG.flush(); rprint(m)


source texts loaded: 952 graph-indexed articles (of 952 graph docs)

## H204 conflict derivation - home-doc-restricted attribute-proximity, disjoint float sets

Products = Device-labeled entities (the product analog on this corpus). Home docs of a product = articles
whose title region (first 300 chars) contains the device name (filename-token path is inert for the opaque
c*_d* article IDs). For each physical attribute, extract values near the keyword within each home doc;
a conflict = a (product, attribute) whose home docs yield disjoint normalized-float value sets.

In [4]:
prod_names = sorted({node[nid]["name"] for nid in node
                     if "Device" in node[nid]["types"] and node[nid]["name"]})
GEN = set("cpap auto pro plus device machine system for her the and with card sd oxygen concentrator portable "
          "stationary medical drive heavy duty precision easy superstar respiratory sleep standard series "
          "brochure user manual datasheet guide care solutions catalog products product bipap apap adjustable "
          "heated humidifier tube nasal mask pillows full face fit pack elite lite".split())
def keyset(nm): return {k for k in (toks(nm) - GEN) if len(k) >= 4}
def home_docs(nm):
    ks = keyset(nm)
    if not ks: return []
    return sorted({dn for dn in DOCS if (ks & toks(dn)) or (_n(nm) in title[dn])})

ATTR = [("weight", "weight", r"\d[\d.,]*\s?(?:kg|lbs?|kilograms?|grams?)\b"),
        ("sound", "sound level", r"\d[\d.,]*\s?dB\s?\(?A?\)?"),
        ("noise", "noise level", r"\d[\d.,]*\s?dB\s?\(?A?\)?"),
        ("pressure", "operating pressure range", r"\d[\d.,]*\s?[-–]\s?\d[\d.,]*\s?(?:cm\s?H\s?2?\s?O|hpa)"),
        ("ramp", "ramp time", r"\d[\d.,]*\s?min(?:ute)?s?\b"),
        ("humidif", "humidifier water capacity", r"\d[\d.,]*\s?(?:ml|millilitres?)\b"),
        ("warrant", "warranty period", r"\d\s?(?:years?|yr)\b"),
        ("dimension", "dimensions", r"\d[\d.,]*\s?[x×]\s?\d[\d.,]*\s?[x×]\s?\d[\d.,]*\s?(?:mm|cm)?"),
        ("power", "power supply", r"\d[\d.,]*\s?(?:w|watts?)\b")]
def norm_floats(val):
    v = val.lower(); v = re.sub(r"(?<=\d),(?=\d{1,2}\b)", ".", v); v = v.replace(",", "")
    return tuple(float(x) for x in re.findall(r"\d+(?:\.\d+)?", v))
UNIT_RE = re.compile(r"(kg|lbs?|grams?|kilograms?|mm|cm|hpa|cmh2o|watts?|w|dba?|min|minutes?|mins|ml|years?|yr)", re.I)
def unit_of(val): return set(UNIT_RE.findall(val.lower().replace(" ", "")))
def vals_near_attr(txt, akw, vre, win=120):
    low = txt.lower(); out = []
    for am in re.finditer(re.escape(akw), low):
        vm = re.search(vre, txt[am.start():am.start() + win], re.I)
        if vm:
            v = re.sub(r"\s+", " ", vm.group(0)).strip()
            if re.search(r"\d", v): out.append(v)
    return out

prodset = [nm for nm in prod_names if len(nm) >= 5]
multi = {nm: home_docs(nm) for nm in prodset}; multi = {nm: ds for nm, ds in multi.items() if len(ds) >= 2}
log(f"Device products: {len(prod_names)} | with keyset & >=2 home docs: {len(multi)}")

raw = []
for nm, ds in multi.items():
    for akw, lab, vre in ATTR:
        dv = {}
        for dn in ds:
            vs = vals_near_attr(rawd[dn], akw, vre)
            if vs: dv[dn] = {norm_floats(v): v for v in vs}
        if len(dv) < 2: continue
        keys = list(dv); pair = None
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                si, sj = set(dv[keys[i]]), set(dv[keys[j]])
                if si and sj and si.isdisjoint(sj): pair = (keys[i], keys[j]); break
            if pair: break
        if pair:
            v1 = list(dv[pair[0]].values())[0]; v2 = list(dv[pair[1]].values())[0]
            raw.append(dict(id=f"C{len(raw)+1:04d}", product=nm, attribute=lab,
                            values=[v1, v2], doc_pair=[pair[0], pair[1]],
                            unit_variant=(unit_of(v1) != unit_of(v2)),
                            genuine_value_conflict=(unit_of(v1) == unit_of(v2))))
log(f"raw conflict golds (per Device name): {len(raw)}")

/tmp/ipykernel_2807372/1338885591.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H210 {m}\n"); LOG.flush(); rprint(m)


Device products: 635 | with keyset & >=2 home docs: 124

raw conflict golds (per Device name): 6

## Alias-collapse -> distinct physical conflicts

In [5]:
seen, distinct = set(), []
for c in raw:
    key = (tuple(sorted(x[:24] for x in c["doc_pair"])), c["attribute"], tuple(sorted(c["values"])))
    if key not in seen: seen.add(key); distinct.append(c)
log(f"distinct physical conflicts (post alias-collapse): {len(distinct)}")

/tmp/ipykernel_2807372/1338885591.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H210 {m}\n"); LOG.flush(); rprint(m)


distinct physical conflicts (post alias-collapse): 4

## H208 dimension-aware unit conversion - drop reconcilable unit-variant pairs (post unit-conversion)

In [6]:
XUNIT = {
 "g": ("mass", 1.0), "kg": ("mass", 1000.0), "oz": ("mass", 28.349523125),
 "lb": ("mass", 453.59237), "lbs": ("mass", 453.59237), "pound": ("mass", 453.59237), "pounds": ("mass", 453.59237),
 "mm": ("len", 1.0), "cm": ("len", 10.0), "inch": ("len", 25.4), "inches": ("len", 25.4), "in": ("len", 25.4),
 "cmh2o": ("press", 1.0), "hpa": ("press", 1.0197162), "mbar": ("press", 1.0197162), "kpa": ("press", 10.197162),
 "l/min": ("flow", 1.0), "lpm": ("flow", 1.0), "ml/min": ("flow", 0.001)}
TOL = 0.02
_XALT = r"(cmh2o|cm h2o|ml/min|l/min|mm|cm|kg|lbs|lb|pounds|pound|oz|inches|inch|hpa|mbar|kpa|lpm|in|g|m)"
def _close(a, b): return b != 0 and abs(a - b) / abs(b) <= TOL
def x_scalar(v):
    g = gnorm(v)
    if re.search(r"\d+\s*x\s*\d+\s*x\s*\d+", g): return None
    out = []
    for m in re.finditer(r"(\d+(?:\.\d+)?)\s*" + _XALT + r"\b", g):
        u = m.group(2).replace(" ", "")
        if u in XUNIT:
            dim, f = XUNIT[u]; out.append((round(float(m.group(1)) * f, 4), dim, u))
    return out or None
def x_dims(v):
    g = gnorm(v)
    mt = re.search(r"(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)\s*" + _XALT + r"?\b", g)
    if not mt: return None
    return [float(mt.group(i)) for i in (1, 2, 3)], (mt.group(4) or "").replace(" ", "")
def values_consistent(v1, v2):
    """True iff v1,v2 are the same physical quantity under conversion (H208)."""
    d1, d2 = x_dims(v1), x_dims(v2)
    if d1 and d2:
        (t1, u1), (t2, u2) = d1, d2
        f1 = XUNIT.get(u1, ("len", 25.4))[1] or 25.4
        f2 = XUNIT.get(u2, ("len", 25.4))[1] or 25.4
        b1 = sorted(x * f1 for x in t1); b2 = sorted(x * f2 for x in t2)
        return all(_close(a, b) for a, b in zip(b1, b2))
    s1, s2 = x_scalar(v1), x_scalar(v2)
    if s1 and s2 and len(s1) == 1 and len(s2) == 1:
        (b1, dim1, _), (b2, dim2, _) = s1[0], s2[0]
        return dim1 == dim2 and _close(b1, b2)
    return None

for c in distinct:
    c["reconciled_by_conversion"] = bool(values_consistent(c["values"][0], c["values"][1]) is True)
genuine = [c for c in distinct if not c["reconciled_by_conversion"]]
n_reconciled = len(distinct) - len(genuine)
log(f"post unit-conversion: {n_reconciled} reconciled (unit-variant), {len(genuine)} genuine physical conflicts remain")

/tmp/ipykernel_2807372/1338885591.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H210 {m}\n"); LOG.flush(); rprint(m)


post unit-conversion: 0 reconciled (unit-variant), 4 genuine physical conflicts remain

## Clause (b) census - does the graph render carry both values / one / neither?

In [7]:
norm2ids = defaultdict(list)
for nid, nm in names.items(): norm2ids[gnorm(nm)].append(nid)
def resolve(pn):
    g = gnorm(pn)
    if g in norm2ids: return norm2ids[g][0]
    c = [nid for nid, nm in names.items() if g == gnorm(nm)] or [nid for nid, nm in names.items() if g in gnorm(nm)]
    return c[0] if c else None
def _floats(v): return re.findall(r"\d+(?:\.\d+)?", re.sub(r"(?<=\d),(?=\d{1,2}\b)", ".", v.lower()).replace(",", ""))
def all_nums_in(val, blob):
    b = blob.replace(" ", ""); ns = _floats(val)
    return bool(ns) and all(re.search(r"(?<!\d)" + re.escape(n) + r"(?!\d)", b) for n in ns)

def census_of(conflicts):
    part = Counter(); rows = []
    for c in conflicts:
        pid = resolve(c["product"])
        gtx = gnorm(seed_render(pid) + " " + " ".join(props_by.get(pid, []))) if pid else ""
        g1 = all_nums_in(c["values"][0], gtx); g2 = all_nums_in(c["values"][1], gtx)
        cs = "both" if (g1 and g2) else ("one" if (g1 or g2) else "neither")
        part[cs] += 1; c["census"] = cs
        rows.append(dict(id=c["id"], product=c["product"], attribute=c["attribute"],
                         values=c["values"], source_articles=c["doc_pair"], census=cs))
    return part, rows

part_g, rows_g = census_of(genuine)
tot_g = max(1, len(genuine))
silent_drop_g = (part_g.get("one", 0) + part_g.get("neither", 0)) / tot_g
part_d, _ = census_of(distinct)
silent_drop_d = (part_d.get("one", 0) + part_d.get("neither", 0)) / max(1, len(distinct))
log(f"census (genuine conflicts, n={len(genuine)}): both={part_g.get('both',0)} one={part_g.get('one',0)} "
    f"neither={part_g.get('neither',0)}  -> silent-drop={silent_drop_g:.3f}")
log(f"census (all distinct, n={len(distinct)}): silent-drop={silent_drop_d:.3f}")

/tmp/ipykernel_2807372/1338885591.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H210 {m}\n"); LOG.flush(); rprint(m)


census (genuine conflicts, n=4): both=1 one=0 neither=3  -> silent-drop=0.750

census (all distinct, n=4): silent-drop=0.750

## Clause verdicts + conflict census report

In [8]:
a_pass = len(genuine) >= 40
b_pass = silent_drop_g >= 0.60
if a_pass:
    verdict = f"CONFIRMED: {len(genuine)} genuine physical conflicts (>= 40) post alias-collapse + unit-conversion; " \
              f"silent-drop {silent_drop_g:.1%} ({'>=' if b_pass else '<'} 60%). " \
              "Version diversity manifests at scale; R17 reconciliation layer is load-bearing"
else:
    verdict = f"REFUTED: only {len(genuine)} genuine physical conflicts (< 40) even at 952-doc scale; " \
              "single-sourcing is a corpus-class property, R17 conflict-reconciliation priority drops. " \
              f"(silent-drop among the {len(genuine)} found = {silent_drop_g:.1%})"
log(f"\nclause (a) >= 40 genuine conflicts: {'PASS' if a_pass else 'FAIL'} ({len(genuine)})")
log(f"clause (b) silent-drop >= 60%: {'PASS' if b_pass else 'FAIL'} ({silent_drop_g:.1%})")
log(f"VERDICT: {verdict}")

log("\ncensus (genuine conflicts, per-conflict evidence):")
for r in rows_g[:60]:
    log(f"   [{r['census']:7}] {r['product'][:24]:24} {r['attribute']:24} "
        f"{r['values'][0]!r} vs {r['values'][1]!r}  <- {r['source_articles']}")

report = dict(round="R19-H210", utc=STAMP, graph="neo4j3", uri=NEO4J3, fingerprint=FP,
              derivation="home-doc-restricted (Device products, title-region match) attribute-proximity; "
                         "disjoint normalized-float value sets across >=2 graph-indexed articles",
              source_texts=len(DOCS), n_products_multidoc=len(multi),
              n_raw_golds=len(raw), n_distinct_physical=len(distinct),
              n_reconciled_by_conversion=n_reconciled, n_genuine_conflicts=len(genuine),
              clause_a_ge40=bool(a_pass), clause_b_silent_drop_ge60=bool(b_pass),
              silent_drop_rate_genuine=round(silent_drop_g, 4), silent_drop_rate_distinct=round(silent_drop_d, 4),
              census_genuine={"both": part_g.get("both", 0), "one": part_g.get("one", 0),
                              "neither": part_g.get("neither", 0)},
              census_distinct={"both": part_d.get("both", 0), "one": part_d.get("one", 0),
                               "neither": part_d.get("neither", 0)},
              conflicts=rows_g, verdict=verdict,
              deviations=["graph is 952 docs / 4649 entities / 26703 rels vs setup message's 471 / 7903 / 6542",
                          "no Proposition nodes and only 2 SAME_AS edges on this graph (props channel empty; "
                          "alias-collapse is the doc-pair/attr/value dedup, not SAME_AS-driven)",
                          "recall@8 retrieval arm of H204 dropped (needs Bedrock query embeddings, banned); "
                          "clauses (a)/(b) are text-derivation + graph-render census, retrieval-free",
                          "both wave dirs used (graph spans c34+c77); restricted to graph-indexed docs"])
out = ROOT / f"reports/conflict-scale-h210-{STAMP}.json"
out.write_text(json.dumps(report, indent=2))
log(f"\nsaved {out}")
LOG.close()

/tmp/ipykernel_2807372/1338885591.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  def log(m): LOG.write(f"[{datetime.datetime.utcnow().isoformat()}] H210 {m}\n"); LOG.flush(); rprint(m)


clause (a) >= 40 genuine conflicts: FAIL (4)

clause (b) silent-drop >= 60%: PASS (75.0%)

VERDICT: REFUTED: only 4 genuine physical conflicts (< 40) even at 952-doc scale; single-sourcing is a corpus-class
property, R17 conflict-reconciliation priority drops. (silent-drop among the 4 found = 75.0%)

census (genuine conflicts, per-conflict evidence):

AirSense 10              operating pressure range '10-20 cm H2O' vs '4-15 cm H2O'  <- ['c34_d22090.txt', 
'c34_d5983.txt']

CPAP Therapy             operating pressure range '4-14 cmH2O' vs '6-7 cmH2O'  <- ['c34_d12607.txt', 
'c34_d15079.txt']

Continuous Positive Airw operating pressure range '9.4-15 cm H2O' vs '6-7 cmH2O'  <- ['c34_d15027.txt', 
'c34_d15079.txt']

Pressurized air          operating pressure range '7-15 cmH2O' vs '4-20 cm H2O'  <- ['c34_d27227.txt', 
'c34_d8203.txt']

saved ../reports/conflict-scale-h210-20260707T205552Z.json